In [1]:
!pip install ultralytics opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [2]:
from google.colab import files

uploaded = files.upload()

Saving cvvido.mp4 to cvvido.mp4


In [3]:
import os

print(os.listdir())

['.config', 'cvvido.mp4', 'sample_data']


In [4]:
from ultralytics import YOLO
import cv2
import numpy as np

# Load YOLO Pose Model
model = YOLO("yolo26n-pose.pt")

# Open your original video
cap = cv2.VideoCapture("cvvido.mp4")

if not cap.isOpened():
    raise ValueError("Video not found! Check video filename.")

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 25

# Create output video
out = cv2.VideoWriter(
    "pose_result.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

# Body keypoint connections
LINKS = [
    (0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),
    (6,8),(8,10),(5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Detect human pose
    results = model(frame, conf=0.5, verbose=False)

    # Black background
    canvas = np.zeros((h, w, 3), dtype=np.uint8)

    keypoints = results[0].keypoints

    if keypoints is not None:

        for person in keypoints.xy.cpu().numpy():

            # Draw skeleton lines
            for a, b in LINKS:

                xa, ya = person[a]
                xb, yb = person[b]

                if xa > 0 and ya > 0 and xb > 0 and yb > 0:

                    cv2.line(
                        canvas,
                        (int(xa), int(ya)),
                        (int(xb), int(yb)),
                        (255, 255, 255),
                        3
                    )

            # Draw keypoints
            for x, y in person:

                if x > 0 and y > 0:

                    cv2.circle(
                        canvas,
                        (int(x), int(y)),
                        15,
                        (255, 255, 255),
                        -1
                    )

    out.write(canvas)

cap.release()
out.release()

print("Pose estimation completed!")
print("Output saved as pose_result.mp4")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Pose estimation completed!
Output saved as pose_result.mp4


In [5]:
from IPython.display import Video, display

display(Video("pose_result.mp4", embed=True))

In [6]:
from google.colab import files

files.download("pose_result.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>